<a href="https://colab.research.google.com/github/Prajwala15/Prajwala15/blob/main/Reading_Text_in_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reading Text in Images for Visual Question Answering (TextVQA)

**Team notebook** — all members run their work here in order.

| Member | Section | What you run |
|--------|---------|--------------|
| **Member 1** | OCR pipeline | OCR cache, OCR-first baseline (Flan-T5), error analysis |
| **Member 2** | VLM pipeline | BLIP-2 answers directly from image + question |
| **Member 3** | Hybrid + analysis | VLM + OCR hint, cross-approach comparison |

**Before you start:** Runtime → **GPU** (T4 is fine). Run **Shared setup** first, then your member section.

Use `LIMIT = 200` for a quick test; `LIMIT = 0` for the full val split (5000 questions).

## Shared setup (all members)

In [ ]:
# Clone repo (skip if you uploaded the folder or already cloned)
import os
if os.path.exists('/content'):
    %cd /content
if not os.path.exists('scripts/01_run_ocr.py'):
    !git clone -q https://github.com/Prajwala15/Prajwala15.git textvqa
    %cd textvqa
else:
    print('Using project folder:', os.getcwd())
!ls -la

In [ ]:
# Install dependencies
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng > /dev/null
!pip install -q -r requirements.txt
!pip install -q paddlepaddle-gpu==2.6.2

In [ ]:
# Download TextVQA data + fix image path in config
import os, yaml, glob
os.makedirs('data', exist_ok=True)
!wget -q -nc -P data https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
!wget -q -nc -P data https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json

if not glob.glob('data/train_images/**/*.jpg', recursive=True):
    !wget -q -nc -O data/train_val_images.zip https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip
    !mkdir -p data/train_images
    !unzip -q data/train_val_images.zip -d data/train_images

candidates = ['data/train_images/train_images', 'data/train_images']
img_dir = next((p for p in candidates if len(glob.glob(p + '/*.jpg')) > 100), candidates[0])
with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['images']['train'] = img_dir
cfg['paths']['images']['val'] = img_dir
with open('configs/default.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Image dir:', img_dir, '| jpg count:', len(glob.glob(img_dir + '/*.jpg')))

In [ ]:
# Global run settings (change here for quick test vs full val)
LIMIT = 200          # 0 = full val (5000); 200 = quick Colab test
ENGINE = 'easyocr'   # easyocr | paddleocr | tesseract  (Member 1)
PREPROCESS = True    # denoise + contrast + deskew
GPU = True

limit_flag = f'--limit {LIMIT}' if LIMIT else ''
pre_flag = '--preprocess' if PREPROCESS else ''
gpu_flag = '--gpu' if GPU else ''

from src.dataset import TextVQADataset
ds = TextVQADataset('val', limit=3)
print('val questions:', len(TextVQADataset('val')))
print('sample question:', ds[0]['question'])
print('sample image size:', ds[0]['image'].size)

---
## Member 1 — OCR pipeline & text extraction

OCR engines, preprocessing, bbox/conf/tokens, OCR+question fusion → Flan-T5, baseline metrics, error analysis.

### 1.1 Run OCR (cache tokens for the team)

In [25]:
path = "src/ocr.py"
text = open(path).read()
old = '"bbox": [min(xs), min(ys), max(xs), max(ys)],'
new = '"bbox": [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))],'
if old not in text:
    raise RuntimeError("Pattern not found — open src/ocr.py and edit manually")
open(path, "w").write(text.replace(old, new))
print("Patched", path)

RuntimeError: Pattern not found — open src/ocr.py and edit manually

In [ ]:
!python scripts/01_run_ocr.py --split val --engine {ENGINE} {limit_flag} {pre_flag} {gpu_flag}

In [ ]:
# Inspect OCR output: text, confidence, bounding boxes
import json
from src.config import load_config, ocr_cache_path
cfg = load_config()
cfg['ocr']['preprocess'] = PREPROCESS
path = ocr_cache_path(cfg, 'val', ENGINE)
with open(path) as f:
    cache = json.load(f)
sample_id = next(iter(cache))
print('cache:', path)
print('image_id:', sample_id)
print('sample tokens:', cache[sample_id][:5])
print('images with text:', sum(1 for v in cache.values() if v))

### 1.2 OCR-first baseline (OCR + question → Flan-T5)

In [ ]:
import yaml
from src.config import load_config
with open('configs/default.yaml') as f:
    disk = yaml.safe_load(f)
disk['ocr']['preprocess'] = PREPROCESS
disk['ocr']['engine'] = ENGINE
with open('configs/default.yaml', 'w') as f:
    yaml.dump(disk, f, default_flow_style=False)
!python scripts/02_eval.py --approach ocr_first --split val {limit_flag} --engine {ENGINE} {pre_flag}

In [ ]:
import json
with open('outputs/scores/ocr_first_val.json') as f:
    print(json.dumps(json.load(f), indent=2))

### 1.3 Low-confidence OCR error analysis

In [ ]:
!python scripts/04_ocr_error_analysis.py --split val --engine {ENGINE} {pre_flag} --conf-threshold 0.5

In [ ]:
# Member 1 deliverables: predictions table + low-confidence tokens
import json, pandas as pd
from IPython.display import display
rows = json.load(open('outputs/preds/ocr_first_val.json'))
df = pd.DataFrame(rows)[['question', 'ocr_text', 'pred', 'vqa_acc', 'f1', 'ocr_bucket']]
display(df.head(10))
low = pd.read_csv('outputs/member1/low_confidence_tokens.csv')
print('Low-confidence tokens:', len(low))
display(low.head(10))

# Preprocessing before/after (one image)
from src.preprocess import preprocess_image
import matplotlib.pyplot as plt
from src.dataset import TextVQADataset
raw = TextVQADataset('val', limit=1)[0]['image']
proc = preprocess_image(raw)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(raw); ax[0].set_title('Original')
ax[1].imshow(proc); ax[1].set_title('Preprocessed')
plt.show()

---
## Member 2 — Vision-language model (VLM)

Answer directly from **image + question** using BLIP-2 (no OCR at inference).

**Note:** Member 1 OCR cache is not required for this section.

In [ ]:
!python scripts/02_eval.py --approach vlm --split val {limit_flag}

In [ ]:
import json, pandas as pd
from IPython.display import display
with open('outputs/scores/vlm_val.json') as f:
    print('VLM scores:', json.dumps(json.load(f), indent=2))
rows = json.load(open('outputs/preds/vlm_val.json'))
display(pd.DataFrame(rows)[['question', 'pred', 'vqa_acc', 'f1', 'ocr_bucket']].head(10))

---
## Member 3 — Hybrid pipeline & cross-approach analysis

Hybrid uses **OCR tokens as a hint** to the VLM. Requires **Member 1 OCR cache**.

Then compare all three approaches by OCR quality bucket.

In [ ]:
import yaml
from src.config import load_config
with open('configs/default.yaml') as f:
    disk = yaml.safe_load(f)
disk['ocr']['preprocess'] = PREPROCESS
disk['ocr']['engine'] = ENGINE
with open('configs/default.yaml', 'w') as f:
    yaml.dump(disk, f, default_flow_style=False)
!python scripts/02_eval.py --approach hybrid --split val {limit_flag} --engine {ENGINE} {pre_flag}

In [ ]:
import json
with open('outputs/scores/hybrid_val.json') as f:
    print('Hybrid scores:', json.dumps(json.load(f), indent=2))

### 3.1 Compare OCR-first vs VLM vs Hybrid

In [ ]:
!python scripts/03_analysis.py --split val

In [ ]:
import pandas as pd
from IPython.display import display, Image
df = pd.read_csv('outputs/comparison_val.csv')
display(df)
display(Image(filename='outputs/comparison_val.png'))